In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)  # long for CrossEntropyLoss
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)



In [ ]:
# 2. Create TensorDataset objects


In [ ]:
# 3. Create DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)





In [ ]:
# 4. Print shape of one batch
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")



In [ ]:
# 5. Display sample images



In [ ]:
# Task 1: Write your model class here:
input_dim = X_train_scaled.shape[1]  # Number of features (columns)
# num_classes already set in STEP 3

print(f"============= AUTOMATIC DIMENSIONS =============")
print(f"Input dim (features): {input_dim}")
print(f"Output dim (classes): {num_classes}")
print(f"================================================")

class TabularClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TabularClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)      # input_dim -> 128
        self.fc2 = nn.Linear(128, 64)             # 128 -> 64
        self.fc3 = nn.Linear(64, num_classes)     # 64 -> num_classes

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # No activation (CrossEntropyLoss handles it)
        return x

model = TabularClassifier(input_dim, num_classes)
print(f"\nModel architecture:")
print(model)

In [ ]:
# Task 2: Write your training loop here:
num_epochs = 50
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    if (epoch + 1) % 10 == 0:
        val_acc = validate(model, criterion, test_loader, device)
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {train_loss:.4f}, Acc: {val_acc:.4f}")

In [ ]:
# Task 3: Write your validation loop here:


In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:

# Train & Validate functions
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)

def validate(model, criterion, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total


In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: